In [24]:
# ============================================================
# D5 — Stage 4 Validation — Branch C: Deterministic Normalisation
# 0. Imports and frozen D5 validation configuration
# ============================================================

from google.colab import files
from pathlib import Path

import hashlib
import json
import math
import re
import unicodedata

import numpy as np
import pandas as pd

from difflib import SequenceMatcher
from scipy.optimize import linear_sum_assignment

DOCUMENT_ID = "D5"
BRANCH_ID = "C"
BRANCH_NAME = "Deterministic normalisation"
PARENT_BRANCH = "B"

EXPECTED_REFERENCE_RECORD_COUNT = 44

EXPECTED_CATEGORY_COUNTS = {
    "Main policy measure": 4,
    "Country profile": 6,
    "Narrative quantitative observation": 16,
    "Statistical table observation": 18
}

EXPECTED_FIELDS = [
    "Category",
    "Indicator or Policy Area",
    "Value",
    "Unit",
    "Qualifier",
    "Reference Period",
    "Description",
    "Source Location"
]

# Frozen from D5 Validation A/B.
PRIMARY_CORRECTNESS_FIELDS = [
    "Category",
    "Indicator or Policy Area",
    "Value",
    "Unit",
    "Qualifier",
    "Reference Period",
    "Source Location"
]

REFERENCE_FIELDS = EXPECTED_FIELDS.copy()

MATCHING_BLOCK_FIELDS = [
    "Category",
    "Source Location"
]

# Frozen fallback thresholds from D5 Branch A.
INDICATOR_MATCH_THRESHOLD = 0.50
DESCRIPTION_MATCH_THRESHOLD = 0.35
FALLBACK_TOTAL_THRESHOLD = 0.45

DESCRIPTION_CORRECTNESS_THRESHOLD = 0.70
NUMERIC_TOLERANCE = 1e-9

OUTPUT_DIR = Path("outputs_D5_validation_branch_C")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("D5 Branch C validation configured.")

D5 Branch C validation configured.


In [25]:
# ------------------------------------------------------------
# 1. Upload validation inputs
# ------------------------------------------------------------
# Required:
#   1) D5_reference_values.csv                  [Stage 1]
#   2) D5_branch_C_parsed_extraction.json      [Branch C]
#   3) D5_branch_C_structure_check.json         [Branch C]
#   4) D5_branch_C_normalisation_check.json     [Branch C]
#
# Do NOT upload a branch-specific reference dataset.
# The Stage 1 reference remains authoritative.

uploaded = files.upload()
uploaded_files = list(uploaded.keys())

csv_files = [
    f for f in uploaded_files
    if f.lower().endswith(".csv")
]

json_files = [
    f for f in uploaded_files
    if f.lower().endswith(".json")
]

if len(csv_files) != 1:
    raise ValueError(
        "Upload exactly one Stage 1 reference CSV."
    )

if len(json_files) != 3:
    raise ValueError(
        "Upload exactly three JSON files: parsed extraction, "
        "structure check, and normalisation check."
    )

REFERENCE_FILE = csv_files[0]
PARSED_EXTRACTION_FILE = None
STRUCTURE_CHECK_FILE = None
NORMALISATION_CHECK_FILE = None

for file_name in json_files:

    with open(
        file_name,
        "r",
        encoding="utf-8-sig"
    ) as f:
        obj = json.load(f)

    if (
        isinstance(obj, dict)
        and obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH_ID
        and isinstance(obj.get("records"), list)
    ):
        PARSED_EXTRACTION_FILE = file_name

    if (
        isinstance(obj, dict)
        and obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH_ID
        and "schema_validity" in obj
        and "records_with_structure_issues" in obj
        and "observed_record_count" in obj
    ):
        STRUCTURE_CHECK_FILE = file_name

    if (
        isinstance(obj, dict)
        and obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH_ID
        and obj.get("parent_branch") == PARENT_BRANCH
        and "normalisation_integrity_passed" in obj
        and "parent_equivalence_passed" in obj
    ):
        NORMALISATION_CHECK_FILE = file_name

if PARSED_EXTRACTION_FILE is None:
    raise ValueError(
        "Could not identify D5 Branch C parsed extraction JSON."
    )

if STRUCTURE_CHECK_FILE is None:
    raise ValueError(
        "Could not identify D5 Branch C structure-check JSON."
    )

if NORMALISATION_CHECK_FILE is None:
    raise ValueError(
        "Could not identify D5 Branch C normalisation-check JSON."
    )

print("Reference:", REFERENCE_FILE)
print("Parsed extraction:", PARSED_EXTRACTION_FILE)
print("Structure check:", STRUCTURE_CHECK_FILE)
print("Normalisation check:", NORMALISATION_CHECK_FILE)

Saving D5_branch_C_structure_check.json to D5_branch_C_structure_check (1).json
Saving D5_branch_C_parsed_extraction.json to D5_branch_C_parsed_extraction (1).json
Saving D5_branch_C_normalisation_check.json to D5_branch_C_normalisation_check (1).json
Saving D5_reference_values.csv to D5_reference_values (1).csv
Reference: D5_reference_values (1).csv
Parsed extraction: D5_branch_C_parsed_extraction (1).json
Structure check: D5_branch_C_structure_check (1).json
Normalisation check: D5_branch_C_normalisation_check (1).json


In [26]:
# ------------------------------------------------------------
# 2. Load inputs, verify identity, and preserve provenance
# ------------------------------------------------------------

with open(
    PARSED_EXTRACTION_FILE,
    "r",
    encoding="utf-8-sig"
) as f:
    extraction_json = json.load(f)

with open(
    STRUCTURE_CHECK_FILE,
    "r",
    encoding="utf-8-sig"
) as f:
    structure_check = json.load(f)

with open(
    NORMALISATION_CHECK_FILE,
    "r",
    encoding="utf-8-sig"
) as f:
    normalisation_check = json.load(f)

reference_df_raw = pd.read_csv(
    REFERENCE_FILE,
    dtype=object,
    keep_default_na=False,
    encoding="utf-8-sig"
)

extracted_df_raw = pd.DataFrame(
    extraction_json["records"]
)

for artefact_name, artefact in {
    "parsed extraction": extraction_json,
    "structure check": structure_check,
    "normalisation check": normalisation_check
}.items():

    if artefact.get("document_id") != DOCUMENT_ID:
        raise ValueError(
            f"Unexpected {artefact_name} document_id: "
            f"{artefact.get('document_id')}"
        )

    if artefact.get("branch") != BRANCH_ID:
        raise ValueError(
            f"Unexpected {artefact_name} branch: "
            f"{artefact.get('branch')}"
        )

if normalisation_check.get("parent_branch") != PARENT_BRANCH:
    raise ValueError(
        "The normalisation artefact does not identify Branch B "
        "as the Branch C parent."
    )


def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(8192),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


input_provenance = {
    "reference_file":
        REFERENCE_FILE,

    "reference_sha256":
        sha256_file(REFERENCE_FILE),

    "parsed_extraction_file":
        PARSED_EXTRACTION_FILE,

    "parsed_extraction_sha256":
        sha256_file(PARSED_EXTRACTION_FILE),

    "structure_check_file":
        STRUCTURE_CHECK_FILE,

    "structure_check_sha256":
        sha256_file(STRUCTURE_CHECK_FILE),

    "normalisation_check_file":
        NORMALISATION_CHECK_FILE,

    "normalisation_check_sha256":
        sha256_file(NORMALISATION_CHECK_FILE)
}

print("Reference shape:", reference_df_raw.shape)
print("Extraction shape:", extracted_df_raw.shape)

Reference shape: (44, 8)
Extraction shape: (46, 8)


In [27]:
# ------------------------------------------------------------
# 3. Restore Stage 1 CSV nulls and mixed numeric values
# ------------------------------------------------------------
# This reproduces the Stage 1 / Validation A representation of values.
# It is comparison-only and does not modify the saved Branch C extraction.

def restore_csv_null(value):

    if value is None:
        return None

    if (
        isinstance(value, float)
        and pd.isna(value)
    ):
        return None

    if isinstance(value, str):

        stripped = value.strip()

        if stripped == "":
            return None

        if stripped.casefold() in {
            "none",
            "null",
            "nan"
        }:
            return None

    return value


def restore_mixed_value(value):

    value = restore_csv_null(value)

    if value is None:
        return None

    if not isinstance(value, str):
        return value

    stripped = value.strip()

    normalised = (
        stripped
        .replace(",", "")
        .replace("−", "-")
        .replace("–", "-")
    )

    if re.fullmatch(
        r"-?\d+",
        normalised
    ):
        return int(normalised)

    if re.fullmatch(
        r"-?\d+\.\d+",
        normalised
    ):
        return float(normalised)

    return stripped


reference_df = reference_df_raw.copy(deep=True)

for field in REFERENCE_FIELDS:
    reference_df[field] = (
        reference_df[field]
        .map(restore_csv_null)
    )

reference_df["Value"] = (
    reference_df["Value"]
    .map(restore_mixed_value)
)

print("Stage 1 CSV representation restored.")

Stage 1 CSV representation restored.


In [28]:
# ------------------------------------------------------------
# 4. Verify the fixed Stage 1 reference dataset
# ------------------------------------------------------------

reference_schema_valid = (
    reference_df.columns.tolist()
    == REFERENCE_FIELDS
)

reference_record_count_valid = (
    len(reference_df)
    == EXPECTED_REFERENCE_RECORD_COUNT
)

reference_category_counts = (
    reference_df["Category"]
    .value_counts()
    .to_dict()
)

reference_category_counts_valid = (
    reference_category_counts
    == EXPECTED_CATEGORY_COUNTS
)

if not reference_schema_valid:
    raise ValueError(
        "D5 Stage 1 reference schema is invalid."
    )

if not reference_record_count_valid:
    raise ValueError(
        f"Expected {EXPECTED_REFERENCE_RECORD_COUNT} Stage 1 records; "
        f"found {len(reference_df)}."
    )

if not reference_category_counts_valid:
    raise ValueError(
        "D5 Stage 1 category counts do not match the frozen scope."
    )

print("Reference schema valid:", reference_schema_valid)
print("Reference record count:", len(reference_df))
print("Reference category counts:", reference_category_counts)

Reference schema valid: True
Reference record count: 44
Reference category counts: {'Statistical table observation': 18, 'Narrative quantitative observation': 16, 'Country profile': 6, 'Main policy measure': 4}


In [29]:
# ------------------------------------------------------------
# 5. Reuse Branch C structural/schema diagnostics
# ------------------------------------------------------------
# Record-count agreement and category-count agreement are
# scope/completeness outcomes, NOT components of schema validity.
#
# D5 Branch C may store some diagnostic counts as None when the
# corresponding records structure is not evaluable. Therefore,
# diagnostic conversion must preserve None rather than calling int(None).

def safe_int(value):
    """
    Convert a numeric diagnostic to int while preserving None.

    This is needed because Branch C structure-check artefacts may use
    None for diagnostics that could not be evaluated.
    """
    if value is None:
        return None

    try:
        return int(value)

    except (TypeError, ValueError):
        return None


schema_validity = bool(
    structure_check.get(
        "schema_validity",
        False
    )
)

content_diagnostics = (
    structure_check.get(
        "content_diagnostics",
        {}
    )
    or {}
)


schema_diagnostics = {

    "valid_json":
        bool(
            structure_check.get(
                "valid_json",
                False
            )
        ),

    "top_level_object_valid":
        bool(
            structure_check.get(
                "top_level_object_valid",
                False
            )
        ),

    "document_id_present":
        bool(
            structure_check.get(
                "document_id_present",
                False
            )
        ),

    "document_id_correct":
        bool(
            structure_check.get(
                "document_id_correct",
                False
            )
        ),

    "branch_present":
        bool(
            structure_check.get(
                "branch_present",
                False
            )
        ),

    "branch_correct":
        bool(
            structure_check.get(
                "branch_correct",
                False
            )
        ),

    "records_present":
        bool(
            structure_check.get(
                "records_present",
                False
            )
        ),

    "records_is_list":
        bool(
            structure_check.get(
                "records_is_list",
                False
            )
        ),

    "records_evaluable":
        bool(
            structure_check.get(
                "records_evaluable",
                False
            )
        ),

    "records_with_structure_issues":
        safe_int(
            structure_check.get(
                "records_with_structure_issues"
            )
        ),

    "records_with_type_issues":
        safe_int(
            structure_check.get(
                "records_with_type_issues"
            )
        ),

    "record_structure_issues":
        structure_check.get(
            "record_structure_issues"
        ),

    "field_type_issues":
        structure_check.get(
            "field_type_issues"
        ),

    "expected_record_count":
        safe_int(
            structure_check.get(
                "expected_record_count"
            )
        ),

    "observed_record_count":
        safe_int(
            structure_check.get(
                "observed_record_count"
            )
        ),

    "record_count_valid":
        structure_check.get(
            "record_count_valid"
        ),

    "expected_category_counts":
        structure_check.get(
            "expected_category_counts"
        ),

    "observed_category_counts":
        structure_check.get(
            "observed_category_counts"
        ),

    "category_counts_valid":
        structure_check.get(
            "category_counts_valid"
        ),

    "scope_complete":
        bool(
            structure_check.get(
                "scope_complete",
                False
            )
        ),

    "duplicate_record_key_count":
        safe_int(
            content_diagnostics.get(
                "duplicate_record_key_count"
            )
        ),

    "table_record_count":
        safe_int(
            content_diagnostics.get(
                "table_record_count"
            )
        ),

    "table_record_count_valid":
        content_diagnostics.get(
            "table_record_count_valid"
        ),

    "schema_validity":
        schema_validity
}


print(
    "Branch C schema diagnostics:"
)

print(
    json.dumps(
        schema_diagnostics,
        indent=2,
        ensure_ascii=False
    )
)

Branch C schema diagnostics:
{
  "valid_json": true,
  "top_level_object_valid": true,
  "document_id_present": true,
  "document_id_correct": true,
  "branch_present": true,
  "branch_correct": true,
  "records_present": true,
  "records_is_list": true,
  "records_evaluable": true,
  "records_with_structure_issues": 0,
  "records_with_type_issues": 0,
  "record_structure_issues": [],
  "field_type_issues": [],
  "expected_record_count": 44,
  "observed_record_count": 46,
  "record_count_valid": false,
  "expected_category_counts": {
    "Main policy measure": 4,
    "Country profile": 6,
    "Narrative quantitative observation": 16,
    "Statistical table observation": 18
  },
  "observed_category_counts": {
    "Main policy measure": 4,
    "Country profile": 6,
    "Narrative quantitative observation": 18,
    "Statistical table observation": 18
  },
  "category_counts_valid": false,
  "scope_complete": false,
  "duplicate_record_key_count": 0,
  "table_record_count": 18,
  "table_r

In [30]:
# ------------------------------------------------------------
# 6. Reuse Branch C representation-integrity diagnostics
# ------------------------------------------------------------
# These checks describe the deterministic B→C preprocessing step.
# They do NOT measure LLM extraction correctness.

representation_integrity = {
    "parent_branch":
        normalisation_check.get(
            "parent_branch"
        ),

    "parent_equivalence_passed":
        bool(
            normalisation_check.get(
                "parent_equivalence_passed",
                False
            )
        ),

    "page_sequence_preserved":
        bool(
            normalisation_check.get(
                "page_sequence_preserved",
                False
            )
        ),

    "source_block_count_preserved":
        bool(
            normalisation_check.get(
                "source_block_count_preserved",
                False
            )
        ),

    "source_block_sequence_preserved":
        bool(
            normalisation_check.get(
                "source_block_sequence_preserved",
                False
            )
        ),

    "deterministic_representation_verified":
        bool(
            normalisation_check.get(
                "deterministic_representation_verified",
                False
            )
        ),

    "all_expected_components_preserved":
        bool(
            normalisation_check.get(
                "all_expected_components_preserved",
                False
            )
        ),

    "table_heading_preserved":
        bool(
            normalisation_check.get(
                "table_heading_preserved",
                False
            )
        ),

    "all_table_rows_preserved":
        bool(
            normalisation_check.get(
                "all_table_rows_preserved",
                False
            )
        ),

    "table_footnote_preserved":
        bool(
            normalisation_check.get(
                "table_footnote_preserved",
                False
            )
        ),

    "numeric_values_preserved":
        bool(
            normalisation_check.get(
                "numeric_values_preserved",
                False
            )
        ),

    "all_table_values_preserved":
        bool(
            normalisation_check.get(
                "all_table_values_preserved",
                False
            )
        ),

    "complete_source_content_retained":
        bool(
            normalisation_check.get(
                "complete_source_content_retained",
                False
            )
        ),

    "out_of_scope_content_retained":
        bool(
            normalisation_check.get(
                "out_of_scope_content_retained",
                False
            )
        ),

    "semantic_harmonisation_applied":
        bool(
            normalisation_check.get(
                "semantic_harmonisation_applied",
                False
            )
        ),

    "semantic_rewriting_applied":
        bool(
            normalisation_check.get(
                "semantic_rewriting_applied",
                False
            )
        ),

    "unit_conversion_applied":
        bool(
            normalisation_check.get(
                "unit_conversion_applied",
                False
            )
        ),

    "numeric_calculation_applied":
        bool(
            normalisation_check.get(
                "numeric_calculation_applied",
                False
            )
        ),

    "manual_correction_applied":
        bool(
            normalisation_check.get(
                "manual_correction_applied",
                False
            )
        ),

    "reference_values_used_for_transformation":
        bool(
            normalisation_check.get(
                "reference_values_used_for_transformation",
                False
            )
        ),

    "normalisation_integrity_passed":
        bool(
            normalisation_check.get(
                "normalisation_integrity_passed",
                False
            )
        )
}

print(
    json.dumps(
        representation_integrity,
        indent=2,
        ensure_ascii=False
    )
)

if representation_integrity[
    "reference_values_used_for_transformation"
]:
    raise ValueError(
        "Branch C reports reference-value use during transformation, "
        "which violates the experimental design."
    )

if not representation_integrity[
    "normalisation_integrity_passed"
]:
    print(
        "WARNING: Branch C representation integrity did not pass. "
        "The extraction can still be described, but interpretation "
        "must distinguish preprocessing loss from extraction error."
    )

{
  "parent_branch": "B",
  "parent_equivalence_passed": true,
  "page_sequence_preserved": true,
  "source_block_count_preserved": true,
  "source_block_sequence_preserved": true,
  "deterministic_representation_verified": true,
  "all_expected_components_preserved": true,
  "table_heading_preserved": true,
  "all_table_rows_preserved": true,
  "table_footnote_preserved": true,
  "numeric_values_preserved": true,
  "all_table_values_preserved": true,
  "complete_source_content_retained": true,
  "out_of_scope_content_retained": true,
  "semantic_harmonisation_applied": false,
  "semantic_rewriting_applied": false,
  "unit_conversion_applied": false,
  "numeric_calculation_applied": false,
  "manual_correction_applied": false,
  "reference_values_used_for_transformation": false,
  "normalisation_integrity_passed": true
}


In [31]:
# ------------------------------------------------------------
# 7. Verify extraction fields without modifying the preserved output
# ------------------------------------------------------------

missing_extraction_columns = [
    field
    for field in EXPECTED_FIELDS
    if field not in extracted_df_raw.columns
]

extracted_df = (
    extracted_df_raw
    .copy(deep=True)
)

# Missing fields are represented as None only in the validation copy.
# Schema validity remains determined by the Branch C structure-check artefact.
for field in missing_extraction_columns:
    extracted_df[field] = None

extracted_df = extracted_df[
    EXPECTED_FIELDS
].copy()

print("Extracted records:", len(extracted_df))
print("Missing extraction columns:", missing_extraction_columns)

Extracted records: 46
Missing extraction columns: []


In [32]:
# ------------------------------------------------------------
# 8. Frozen comparison normalisation from D5 Branch A/B
# ------------------------------------------------------------
# These operations are Stage 4 comparison rules only.
# They do not change the raw Branch C extraction.

DASH_REPLACEMENTS = {
    "\u2010": "-",
    "\u2011": "-",
    "\u2012": "-",
    "\u2013": "-",
    "\u2014": "-",
    "\u2212": "-"
}

APOSTROPHE_REPLACEMENTS = {
    "\u2018": "'",
    "\u2019": "'",
    "\u02bc": "'",
    "`": "'"
}


def normalise_text(value):

    if (
        value is None
        or (
            isinstance(value, float)
            and pd.isna(value)
        )
    ):
        return None

    text = unicodedata.normalize(
        "NFKC",
        str(value)
    )

    text = (
        text
        .replace("\u00a0", " ")
        .replace("\u2007", " ")
        .replace("\u202f", " ")
    )

    for source, target in DASH_REPLACEMENTS.items():
        text = text.replace(
            source,
            target
        )

    for source, target in APOSTROPHE_REPLACEMENTS.items():
        text = text.replace(
            source,
            target
        )

    text = re.sub(
        r"[ \t\f\v]+",
        " ",
        text
    )

    text = re.sub(
        r" *\n *",
        "\n",
        text
    )

    return (
        text
        .strip()
        .casefold()
    )


INDICATOR_EQUIVALENCE_RAW = {
    "Price and financial stability": [
        "Managing price and financial stability",
    ],
    "Fiscal sustainability while financing development": [
        "Achieving fiscal sustainability while financing development",
    ],
    "Financial sector development": [
        "Broadening and deepening the financial sector",
    ],
    "Job-intensive inclusive growth": [
        "Promoting more job-intensive, inclusive growth",
    ],
    "Recent wholesale price inflation": [
        "Wholesale price index annual rise",
    ],
    "Wholesale price inflation one year earlier": [
        "Wholesale price index annual rise",
    ],
    "Previous high general government deficit": [
        "General government deficit high",
    ],
    "Agricultural labour-force share": [
        "Labor force working in agriculture",
    ],
    "Population living on less than USD 2 a day": [
        "Indians living on less than $2 a day",
    ],
}


INDICATOR_EQUIVALENCE = {
    normalise_text(reference): {
        normalise_text(variant)
        for variant in variants
    }
    for reference, variants
    in INDICATOR_EQUIVALENCE_RAW.items()
}


def indicator_equivalent(
    reference_value,
    extracted_value
):

    ref = normalise_text(
        reference_value
    )

    ext = normalise_text(
        extracted_value
    )

    if ref == ext:
        return True

    return (
        ext
        in INDICATOR_EQUIVALENCE.get(
            ref,
            set()
        )
    )


STOPWORDS = {
    "a","an","and","as","at","by","for","from","in","into","is","of",
    "on","or","the","to","was","were","with","reported","reporting",
    "described","principal","measure","observation","india","indias"
}


def comparison_tokens(value):

    text = normalise_text(value)

    if text is None:
        return set()

    text = re.sub(
        r"[^a-z0-9]+",
        " ",
        text
    )

    return {
        token
        for token in text.split()
        if (
            token
            and token not in STOPWORDS
        )
    }


def text_similarity(
    first_value,
    second_value
):

    first_text = normalise_text(
        first_value
    )

    second_text = normalise_text(
        second_value
    )

    if (
        first_text is None
        and second_text is None
    ):
        return 1.0

    if (
        first_text is None
        or second_text is None
    ):
        return 0.0

    if first_text == second_text:
        return 1.0

    first_tokens = comparison_tokens(
        first_value
    )

    second_tokens = comparison_tokens(
        second_value
    )

    if (
        first_tokens
        or second_tokens
    ):
        token_score = (
            len(
                first_tokens
                & second_tokens
            )
            /
            len(
                first_tokens
                | second_tokens
            )
        )
    else:
        token_score = 0.0

    sequence_score = (
        SequenceMatcher(
            None,
            first_text,
            second_text
        ).ratio()
    )

    return max(
        token_score,
        sequence_score
    )


def numeric_value(value):

    if (
        value is None
        or isinstance(value, bool)
    ):
        return None

    if isinstance(
        value,
        (
            int,
            float,
            np.integer,
            np.floating
        )
    ):
        return float(value)

    text = str(value).strip()

    text = (
        text
        .replace(",", "")
        .replace("$", "")
        .replace("–", "-")
        .replace("−", "-")
    )

    if re.fullmatch(
        r"-?\d+(?:\.\d+)?",
        text
    ):
        return float(text)

    return None


def values_match(
    reference_value,
    extracted_value
):

    ref_num = numeric_value(
        reference_value
    )

    ext_num = numeric_value(
        extracted_value
    )

    if (
        ref_num is not None
        and ext_num is not None
    ):

        return math.isclose(
            ref_num,
            ext_num,
            rel_tol=0.0,
            abs_tol=NUMERIC_TOLERANCE
        )

    return (
        normalise_text(
            reference_value
        )
        ==
        normalise_text(
            extracted_value
        )
    )


UNIT_EQUIVALENCE_MAP = {
    "usd":
        "usd",

    "dollars":
        "usd",

    "dollar":
        "usd",

    "billion":
        "billion people",

    "billion people":
        "billion people",

    "million people":
        "million people",

    "million indians":
        "million people",

    "usd billion":
        "billion dollars",

    "billion dollars":
        "billion dollars",

    "percent of labor force":
        "percent of labour force",

    "percent of the labor force":
        "percent of labour force",

    "percent of labour force":
        "percent of labour force",

    "percent of the labour force":
        "percent of labour force"
}


def normalise_unit(value):

    norm = normalise_text(
        value
    )

    if norm is None:
        return None

    return UNIT_EQUIVALENCE_MAP.get(
        norm,
        norm
    )


def units_match(
    reference_value,
    extracted_value
):

    return (
        normalise_unit(
            reference_value
        )
        ==
        normalise_unit(
            extracted_value
        )
    )


def normalise_qualifier(value):

    norm = normalise_text(
        value
    )

    if norm is None:
        return None

    return norm.rstrip(".")


def qualifiers_match(
    reference_value,
    extracted_value
):

    return (
        normalise_qualifier(
            reference_value
        )
        ==
        normalise_qualifier(
            extracted_value
        )
    )


FISCAL_PERIOD_PATTERN = re.compile(
    r"\b((?:19|20)\d{2})\s*[/–-]\s*(\d{2})\b"
)


def canonical_relative_period(value):

    text = normalise_text(
        value
    )

    if text is None:
        return None

    match = FISCAL_PERIOD_PATTERN.search(
        str(value)
    )

    if match:
        return (
            f"{match.group(1)}/"
            f"{match.group(2)}"
        )

    equivalence = {
        "four years running":
            "four years running",

        "next 10 years":
            "next 10 years",

        "over the next 10 years":
            "next 10 years",

        "since 2002":
            "since 2002",

        "recently":
            "recently",

        "a year ago":
            "a year ago",

        "one year earlier":
            "a year ago",

        "medium term":
            "medium term",

        "over the medium term":
            "medium term",

        "after the latest tightening":
            "current tightening",

        "just last week":
            "current tightening",

        "now":
            "current tightening",

        "before the latest tightening":
            "previous tightening",

        "previously":
            "previous tightening",

        "earlier":
            "previous tightening"
    }

    return equivalence.get(
        text,
        text
    )


def periods_match(
    reference_value,
    extracted_value
):

    return (
        canonical_relative_period(
            reference_value
        )
        ==
        canonical_relative_period(
            extracted_value
        )
    )

In [33]:
# ------------------------------------------------------------
# 9. Prepare source-grounded matching blocks
# ------------------------------------------------------------
# Frozen from D5 Validation A/B:
#   block = Category + Source Location
#
# Value and Unit are explicitly excluded from identity.

reference_cmp = (
    reference_df
    .copy(deep=True)
)

extraction_cmp = (
    extracted_df
    .copy(deep=True)
)

for df in (
    reference_cmp,
    extraction_cmp
):

    df["_category_norm"] = (
        df["Category"]
        .map(normalise_text)
    )

    df["_source_norm"] = (
        df["Source Location"]
        .map(normalise_text)
    )

    df["_indicator_norm"] = (
        df["Indicator or Policy Area"]
        .map(normalise_text)
    )

    df["_period_norm"] = (
        df["Reference Period"]
        .map(canonical_relative_period)
    )

    df["_block"] = list(
        zip(
            df["_category_norm"],
            df["_source_norm"]
        )
    )

reference_cmp[
    "_reference_index"
] = range(
    len(reference_cmp)
)

extraction_cmp[
    "_extraction_index"
] = range(
    len(extraction_cmp)
)

print("D5 matching blocks prepared.")

D5 matching blocks prepared.


In [34]:
# ------------------------------------------------------------
# 10. Deterministic one-to-one alignment
# ------------------------------------------------------------
# EXACTLY the Branch-A-frozen D5 strategy:
#
# Stage 1:
#   exact normalised Indicator + canonical Period within the block.
#
# Stage 2:
#   Hungarian fallback using Indicator + Description,
#   with Period only as a weak disambiguation signal.
#
# Value and Unit NEVER influence alignment.

matched_pairs = []
used_ref = set()
used_ext = set()

all_blocks = sorted(
    set(
        reference_cmp["_block"]
    )
    |
    set(
        extraction_cmp["_block"]
    ),
    key=str
)

for block in all_blocks:

    ref_block = (
        reference_cmp[
            reference_cmp["_block"]
            == block
        ]
    )

    ext_block = (
        extraction_cmp[
            extraction_cmp["_block"]
            == block
        ]
    )

    if (
        ref_block.empty
        or ext_block.empty
    ):
        continue

    # ----------------------------------------
    # Stage 1: exact descriptive identity
    # ----------------------------------------

    ext_lookup = {}

    for _, ext_row in ext_block.iterrows():

        key = (
            ext_row[
                "_indicator_norm"
            ],
            ext_row[
                "_period_norm"
            ]
        )

        ext_lookup.setdefault(
            key,
            []
        ).append(
            int(
                ext_row[
                    "_extraction_index"
                ]
            )
        )

    for _, ref_row in ref_block.iterrows():

        ref_idx = int(
            ref_row[
                "_reference_index"
            ]
        )

        key = (
            ref_row[
                "_indicator_norm"
            ],
            ref_row[
                "_period_norm"
            ]
        )

        candidates = [
            idx
            for idx
            in ext_lookup.get(
                key,
                []
            )
            if idx not in used_ext
        ]

        if candidates:

            ext_idx = candidates[0]

            used_ref.add(
                ref_idx
            )

            used_ext.add(
                ext_idx
            )

            matched_pairs.append({
                "reference_index":
                    ref_idx,

                "extraction_index":
                    ext_idx,

                "match_method":
                    "strict_identity",

                "alignment_score":
                    1.0
            })

    # ----------------------------------------
    # Stage 2: descriptive Hungarian fallback
    # ----------------------------------------

    remaining_refs = [
        row
        for _, row
        in ref_block.iterrows()
        if int(
            row[
                "_reference_index"
            ]
        ) not in used_ref
    ]

    remaining_exts = [
        row
        for _, row
        in ext_block.iterrows()
        if int(
            row[
                "_extraction_index"
            ]
        ) not in used_ext
    ]

    if (
        not remaining_refs
        or not remaining_exts
    ):
        continue

    cost_matrix = []
    details_matrix = []

    for ref_row in remaining_refs:

        cost_row = []
        details_row = []

        for ext_row in remaining_exts:

            indicator_score = (
                text_similarity(
                    ref_row[
                        "Indicator or Policy Area"
                    ],
                    ext_row[
                        "Indicator or Policy Area"
                    ]
                )
            )

            description_score = (
                text_similarity(
                    ref_row[
                        "Description"
                    ],
                    ext_row[
                        "Description"
                    ]
                )
            )

            period_score = (
                1.0
                if periods_match(
                    ref_row[
                        "Reference Period"
                    ],
                    ext_row[
                        "Reference Period"
                    ]
                )
                else 0.0
            )

            total = (
                0.70
                * indicator_score

                + 0.25
                * description_score

                + 0.05
                * period_score
            )

            cost_row.append(
                1.0
                - total
            )

            details_row.append({
                "total":
                    total,

                "indicator":
                    indicator_score,

                "description":
                    description_score,

                "period":
                    period_score
            })

        cost_matrix.append(
            cost_row
        )

        details_matrix.append(
            details_row
        )

    row_idx, col_idx = (
        linear_sum_assignment(
            cost_matrix
        )
    )

    for rpos, cpos in zip(
        row_idx,
        col_idx
    ):

        ref_row = (
            remaining_refs[
                rpos
            ]
        )

        ext_row = (
            remaining_exts[
                cpos
            ]
        )

        details = (
            details_matrix[
                rpos
            ][
                cpos
            ]
        )

        if (
            details["total"]
            < FALLBACK_TOTAL_THRESHOLD
        ):
            continue

        if (
            details["indicator"]
            < INDICATOR_MATCH_THRESHOLD

            and details[
                "description"
            ]
            < DESCRIPTION_MATCH_THRESHOLD
        ):
            continue

        ref_idx = int(
            ref_row[
                "_reference_index"
            ]
        )

        ext_idx = int(
            ext_row[
                "_extraction_index"
            ]
        )

        if (
            ref_idx in used_ref
            or ext_idx in used_ext
        ):
            continue

        used_ref.add(
            ref_idx
        )

        used_ext.add(
            ext_idx
        )

        matched_pairs.append({
            "reference_index":
                ref_idx,

            "extraction_index":
                ext_idx,

            "match_method":
                "descriptive_fallback",

            "alignment_score":
                details["total"],

            "indicator_alignment_score":
                details["indicator"],

            "description_alignment_score":
                details["description"],

            "period_disambiguation_score":
                details["period"]
        })


matched_pairs = sorted(
    matched_pairs,
    key=lambda x:
        x["reference_index"]
)

missing_reference_indices = [
    idx
    for idx
    in reference_cmp[
        "_reference_index"
    ].tolist()
    if idx not in used_ref
]

unsupported_extraction_indices = [
    idx
    for idx
    in extraction_cmp[
        "_extraction_index"
    ].tolist()
    if idx not in used_ext
]

print("Aligned records:", len(matched_pairs))
print(
    "Missing reference records:",
    len(missing_reference_indices)
)
print(
    "Unsupported extracted records:",
    len(unsupported_extraction_indices)
)

match_method_counts = (
    pd.Series(
        [
            pair["match_method"]
            for pair
            in matched_pairs
        ],
        dtype="object"
    )
    .value_counts()
)

print("\nMatch methods:")
print(match_method_counts)

Aligned records: 44
Missing reference records: 0
Unsupported extracted records: 2

Match methods:
strict_identity         32
descriptive_fallback    12
Name: count, dtype: int64


In [35]:
# ------------------------------------------------------------
# 11. Compare aligned records field by field
# ------------------------------------------------------------
# Description remains diagnostic only.
# All other fields are the frozen primary correctness fields.

comparison_rows = []
field_rows = []

for pair in matched_pairs:

    ref = (
        reference_cmp.loc[
            reference_cmp[
                "_reference_index"
            ]
            == pair[
                "reference_index"
            ]
        ]
        .iloc[0]
    )

    ext = (
        extraction_cmp.loc[
            extraction_cmp[
                "_extraction_index"
            ]
            == pair[
                "extraction_index"
            ]
        ]
        .iloc[0]
    )

    field_matches = {
        "Category":
            (
                normalise_text(
                    ref["Category"]
                )
                ==
                normalise_text(
                    ext["Category"]
                )
            ),

        "Indicator or Policy Area":
            indicator_equivalent(
                ref[
                    "Indicator or Policy Area"
                ],
                ext[
                    "Indicator or Policy Area"
                ]
            ),

        "Value":
            values_match(
                ref["Value"],
                ext["Value"]
            ),

        "Unit":
            units_match(
                ref["Unit"],
                ext["Unit"]
            ),

        "Qualifier":
            qualifiers_match(
                ref["Qualifier"],
                ext["Qualifier"]
            ),

        "Reference Period":
            periods_match(
                ref["Reference Period"],
                ext["Reference Period"]
            ),

        "Description":
            (
                text_similarity(
                    ref["Description"],
                    ext["Description"]
                )
                >=
                DESCRIPTION_CORRECTNESS_THRESHOLD
            ),

        "Source Location":
            (
                normalise_text(
                    ref["Source Location"]
                )
                ==
                normalise_text(
                    ext["Source Location"]
                )
            )
    }

    all_primary_fields_match = all(
        field_matches[field]
        for field
        in PRIMARY_CORRECTNESS_FIELDS
    )

    comparison_rows.append({
        "Reference Record ID":
            (
                f"D5-REF-"
                f"{pair['reference_index'] + 1:03d}"
            ),

        "Extraction Record ID":
            (
                f"D5-C-"
                f"{pair['extraction_index'] + 1:03d}"
            ),

        "match_method":
            pair["match_method"],

        "alignment_score":
            pair.get(
                "alignment_score"
            ),

        **{
            f"{field}_ref":
                ref[field]
            for field
            in EXPECTED_FIELDS
        },

        **{
            f"{field}_ext":
                ext[field]
            for field
            in EXPECTED_FIELDS
        },

        **{
            f"{field}_match":
                match
            for field, match
            in field_matches.items()
        },

        "all_primary_fields_match":
            all_primary_fields_match,

        "record_status":
            (
                "fully_correct"
                if all_primary_fields_match
                else "discrepant"
            ),

        "all_mismatched_fields":
            ", ".join(
                field
                for field, match
                in field_matches.items()
                if not match
            ),

        "primary_mismatched_fields":
            ", ".join(
                field
                for field
                in PRIMARY_CORRECTNESS_FIELDS
                if not field_matches[field]
            )
    })

    for field in EXPECTED_FIELDS:

        field_rows.append({
            "Reference Record ID":
                (
                    f"D5-REF-"
                    f"{pair['reference_index'] + 1:03d}"
                ),

            "Extraction Record ID":
                (
                    f"D5-C-"
                    f"{pair['extraction_index'] + 1:03d}"
                ),

            "Field":
                field,

            "Field Match":
                field_matches[field],

            "match_method":
                pair["match_method"]
        })


comparison_df = pd.DataFrame(
    comparison_rows
)

field_validation_df = pd.DataFrame(
    field_rows
)

display(
    comparison_df.head()
)

,Reference Record ID,Extraction Record ID,match_method,alignment_score,Category_ref,Indicator or Policy Area_ref,Value_ref,Unit_ref,Qualifier_ref,Reference Period_ref,...,Value_match,Unit_match,Qualifier_match,Reference Period_match,Description_match,Source Location_match,all_primary_fields_match,record_status,all_mismatched_fields,primary_mismatched_fields
0,D5-REF-001,D5-C-001,strict_identity,1.000000,Main policy measure,Price and financial stability,Managing price and financial stability by limi...,text,None,None,...,False,True,True,True,False,True,False,discrepant,"Value, Description",Value
1,D5-REF-002,D5-C-002,descriptive_fallback,0.713139,Main policy measure,Fiscal sustainability while financing development,Achieving fiscal sustainability while financin...,text,None,None,...,False,True,True,True,False,True,False,discrepant,"Indicator or Policy Area, Value, Description","Indicator or Policy Area, Value"
2,D5-REF-003,D5-C-003,strict_identity,1.000000,Main policy measure,Financial sector development,Broadening and deepening the financial sector ...,text,None,None,...,False,True,True,True,False,True,False,discrepant,"Value, Description",Value
3,D5-REF-004,D5-C-004,strict_identity,1.000000,Main policy measure,Job-intensive inclusive growth,"Promoting more job-intensive, inclusive growth...",text,None,None,...,False,True,True,True,False,True,False,discrepant,"Value, Description",Value
4,D5-REF-005,D5-C-005,strict_identity,1.000000,Country profile,Capital,New Delhi,text,None,None,...,True,True,True,True,True,True,True,fully_correct,,


In [36]:
# ------------------------------------------------------------
# 12. Build outcome datasets
# ------------------------------------------------------------

missing_records = (
    reference_cmp[
        reference_cmp[
            "_reference_index"
        ].isin(
            missing_reference_indices
        )
    ]
    .copy()
)

unsupported_records = (
    extraction_cmp[
        extraction_cmp[
            "_extraction_index"
        ].isin(
            unsupported_extraction_indices
        )
    ]
    .copy()
)

fully_correct_records = (
    comparison_df[
        comparison_df[
            "record_status"
        ]
        == "fully_correct"
    ]
    .copy()
)

discrepant_records = (
    comparison_df[
        comparison_df[
            "record_status"
        ]
        == "discrepant"
    ]
    .copy()
)

for df in (
    missing_records,
    unsupported_records
):

    helper_cols = [
        col
        for col in df.columns
        if col.startswith("_")
    ]

    if helper_cols:
        df.drop(
            columns=helper_cols,
            inplace=True
        )

print(
    "Fully correct:",
    len(fully_correct_records)
)
print(
    "Discrepant:",
    len(discrepant_records)
)
print(
    "Missing:",
    len(missing_records)
)
print(
    "Unsupported:",
    len(unsupported_records)
)

Fully correct: 30
Discrepant: 14
Missing: 0
Unsupported: 2


In [37]:
# ------------------------------------------------------------
# 13. Calculate common validation metrics
# ------------------------------------------------------------

N_REF = int(
    len(reference_cmp)
)

N_EXT = int(
    len(extraction_cmp)
)

N_ALIGNED = int(
    len(comparison_df)
)

N_CORRECT = int(
    len(fully_correct_records)
)

N_DISCREPANT = int(
    len(discrepant_records)
)

N_MISSING = int(
    len(missing_records)
)

N_UNSUPPORTED = int(
    len(unsupported_records)
)

completeness = (
    N_ALIGNED / N_REF
    if N_REF
    else 0.0
)

missing_rate = (
    N_MISSING / N_REF
    if N_REF
    else 0.0
)

record_precision = (
    N_CORRECT / N_EXT
    if N_EXT
    else 0.0
)

record_recall = (
    N_CORRECT / N_REF
    if N_REF
    else 0.0
)

record_f1 = (
    2
    * record_precision
    * record_recall
    /
    (
        record_precision
        + record_recall
    )
    if (
        record_precision
        + record_recall
    )
    else 0.0
)

unsupported_rate = (
    N_UNSUPPORTED / N_EXT
    if N_EXT
    else 0.0
)

discrepancy_rate = (
    N_DISCREPANT / N_ALIGNED
    if N_ALIGNED
    else 0.0
)

field_accuracy_among_aligned = {}

for field in EXPECTED_FIELDS:

    rows = (
        field_validation_df[
            field_validation_df[
                "Field"
            ]
            == field
        ]
    )

    field_accuracy_among_aligned[
        field
    ] = (
        float(
            rows[
                "Field Match"
            ].mean()
        )
        if len(rows)
        else 0.0
    )

primary_field_validation_df = (
    field_validation_df[
        field_validation_df[
            "Field"
        ].isin(
            PRIMARY_CORRECTNESS_FIELDS
        )
    ]
)

correct_primary_field_instances = int(
    primary_field_validation_df[
        "Field Match"
    ].sum()
)

expected_primary_field_instances = int(
    N_REF
    * len(
        PRIMARY_CORRECTNESS_FIELDS
    )
)

# Missing expected records therefore contribute incorrect field instances.
overall_primary_field_accuracy = (
    correct_primary_field_instances
    / expected_primary_field_instances
    if expected_primary_field_instances
    else 0.0
)

description_diagnostic_accuracy = (
    field_accuracy_among_aligned[
        "Description"
    ]
)

print("Reference records:", N_REF)
print("Extracted records:", N_EXT)
print("Aligned records:", N_ALIGNED)
print("Fully correct:", N_CORRECT)
print("Discrepant:", N_DISCREPANT)
print("Missing:", N_MISSING)
print("Unsupported:", N_UNSUPPORTED)
print("Completeness:", round(completeness, 4))
print("Exact F1:", round(record_f1, 4))
print(
    "Overall primary-field accuracy:",
    round(
        overall_primary_field_accuracy,
        4
    )
)

Reference records: 44
Extracted records: 46
Aligned records: 44
Fully correct: 30
Discrepant: 14
Missing: 0
Unsupported: 2
Completeness: 1.0
Exact F1: 0.6667
Overall primary-field accuracy: 0.9513


In [38]:
# ------------------------------------------------------------
# 14. Field-level and category-level diagnostics
# ------------------------------------------------------------

field_error_rows = []

for field in EXPECTED_FIELDS:

    rows = (
        field_validation_df[
            field_validation_df[
                "Field"
            ]
            == field
        ]
    )

    correct = int(
        rows[
            "Field Match"
        ].sum()
    )

    incorrect = int(
        len(rows)
        - correct
    )

    field_error_rows.append({
        "field":
            field,

        "is_primary_correctness_field":
            field
            in PRIMARY_CORRECTNESS_FIELDS,

        "aligned_records_evaluated":
            int(
                len(rows)
            ),

        "correct_values_among_aligned":
            correct,

        "incorrect_values_among_aligned":
            incorrect,

        "accuracy_among_aligned":
            (
                round(
                    correct
                    / len(rows),
                    4
                )
                if len(rows)
                else 0.0
            ),

        "missing_expected_instances":
            N_MISSING,

        "overall_expected_instances":
            N_REF,

        "overall_field_accuracy":
            (
                round(
                    correct
                    / N_REF,
                    4
                )
                if N_REF
                else 0.0
            )
    })

field_error_summary_df = (
    pd.DataFrame(
        field_error_rows
    )
)


category_rows = []

for (
    category,
    expected_count
) in EXPECTED_CATEGORY_COUNTS.items():

    ref_indices = set(
        reference_cmp.loc[
            reference_cmp[
                "Category"
            ]
            == category,
            "_reference_index"
        ].tolist()
    )

    aligned_category = (
        comparison_df[
            comparison_df[
                "Reference Record ID"
            ].apply(
                lambda value:
                    int(
                        str(value)
                        .split("-")[-1]
                    )
                    - 1
                    in ref_indices
            )
        ]
    )

    extracted_category_count = int(
        (
            extraction_cmp[
                "Category"
            ]
            == category
        ).sum()
    )

    unsupported_category_count = int(
        (
            unsupported_records[
                "Category"
            ]
            == category
        ).sum()
    ) if not unsupported_records.empty else 0

    fully_correct_count = int(
        (
            aligned_category[
                "record_status"
            ]
            == "fully_correct"
        ).sum()
    )

    discrepant_count = int(
        (
            aligned_category[
                "record_status"
            ]
            == "discrepant"
        ).sum()
    )

    aligned_count = int(
        len(
            aligned_category
        )
    )

    category_rows.append({
        "Category":
            category,

        "Expected Records":
            expected_count,

        "Extracted Records":
            extracted_category_count,

        "Aligned Records":
            aligned_count,

        "Fully Correct Records":
            fully_correct_count,

        "Discrepant Records":
            discrepant_count,

        "Missing Records":
            expected_count
            - aligned_count,

        "Unsupported Records":
            unsupported_category_count,

        "Exact Recall":
            round(
                fully_correct_count
                / expected_count,
                4
            )
            if expected_count
            else 0.0,

        "Exact Precision":
            round(
                fully_correct_count
                / extracted_category_count,
                4
            )
            if extracted_category_count
            else 0.0
    })

category_metrics_df = (
    pd.DataFrame(
        category_rows
    )
)

display(field_error_summary_df)
display(category_metrics_df)

,field,is_primary_correctness_field,aligned_records_evaluated,correct_values_among_aligned,incorrect_values_among_aligned,accuracy_among_aligned,missing_expected_instances,overall_expected_instances,overall_field_accuracy
0,Category,True,44,44,0,1.0000,0,44,1.0000
1,Indicator or Policy Area,True,44,38,6,0.8636,0,44,0.8636
2,Value,True,44,40,4,0.9091,0,44,0.9091
3,Unit,True,44,43,1,0.9773,0,44,0.9773
4,Qualifier,True,44,44,0,1.0000,0,44,1.0000
5,Reference Period,True,44,40,4,0.9091,0,44,0.9091
6,Description,False,44,13,31,0.2955,0,44,0.2955
7,Source Location,True,44,44,0,1.0000,0,44,1.0000


,Category,Expected Records,Extracted Records,Aligned Records,Fully Correct Records,Discrepant Records,Missing Records,Unsupported Records,Exact Recall,Exact Precision
0,Main policy measure,4,4,4,0,4,0,0,0.000,0.0000
1,Country profile,6,6,6,6,0,0,0,1.000,1.0000
2,Narrative quantitative observation,16,18,16,6,10,0,2,0.375,0.3333
3,Statistical table observation,18,18,18,18,0,0,0,1.000,1.0000


In [39]:
# ------------------------------------------------------------
# 15. Build final Branch C validation summary
# ------------------------------------------------------------

summary = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH_ID,

    "branch_name":
        BRANCH_NAME,

    "parent_branch":
        PARENT_BRANCH,

    "reference_records":
        N_REF,

    "extracted_records":
        N_EXT,

    "aligned_records":
        N_ALIGNED,

    "fully_correct_records":
        N_CORRECT,

    "discrepant_records":
        N_DISCREPANT,

    "missing_records":
        N_MISSING,

    "unsupported_records":
        N_UNSUPPORTED,

    "completeness":
        round(
            completeness,
            4
        ),

    "missing_rate":
        round(
            missing_rate,
            4
        ),

    "record_precision_exact":
        round(
            record_precision,
            4
        ),

    "record_recall_exact":
        round(
            record_recall,
            4
        ),

    "record_f1_exact":
        round(
            record_f1,
            4
        ),

    "unsupported_rate":
        round(
            unsupported_rate,
            4
        ),

    "discrepancy_rate_among_aligned":
        round(
            discrepancy_rate,
            4
        ),

    "overall_primary_field_accuracy":
        round(
            overall_primary_field_accuracy,
            4
        ),

    "description_diagnostic_accuracy":
        round(
            description_diagnostic_accuracy,
            4
        ),

    "field_accuracy_among_aligned": {
        key:
            round(
                value,
                4
            )
        for key, value
        in field_accuracy_among_aligned.items()
    },

    "schema_validity":
        schema_validity,

    "schema_diagnostics":
        schema_diagnostics,

    "branch_C_representation_integrity":
        representation_integrity,

    "matching_rules": {
        "blocking_fields":
            MATCHING_BLOCK_FIELDS,

        "strict_identity":
            (
                "Normalised Indicator or Policy Area "
                "+ canonical Reference Period within "
                "Category + Source Location block"
            ),

        "fallback_assignment":
            "Hungarian linear-sum assignment",

        "fallback_fields": [
            "Indicator or Policy Area",
            "Description"
        ],

        "reference_period_role":
            "Weak disambiguation only",

        "fallback_weights": {
            "indicator":
                0.70,

            "description":
                0.25,

            "reference_period":
                0.05
        },

        "indicator_match_threshold":
            INDICATOR_MATCH_THRESHOLD,

        "description_match_threshold":
            DESCRIPTION_MATCH_THRESHOLD,

        "fallback_total_threshold":
            FALLBACK_TOTAL_THRESHOLD,

        "value_used_for_alignment":
            False,

        "unit_used_for_alignment":
            False
    },

    "comparison_rules": {
        "rules_frozen_from_branch_A":
            True,

        "primary_correctness_fields":
            PRIMARY_CORRECTNESS_FIELDS,

        "description_role":
            (
                "Diagnostic only; not part of exact-record correctness"
            ),

        "indicator":
            (
                "Normalised exact comparison plus the same "
                "predefined source-grounded equivalence map "
                "established in Branch A."
            ),

        "value":
            (
                "Numeric equality where numeric; "
                "controlled normalised text otherwise."
            ),

        "unit":
            (
                "Controlled source-grounded equivalence map "
                "frozen in Branch A."
            ),

        "qualifier":
            "Case/punctuation normalisation only.",

        "reference_period":
            (
                "Controlled source-grounded canonical "
                "equivalence frozen in Branch A."
            ),

        "description":
            (
                "Deterministic lexical similarity after alignment; "
                "similarity >= 0.70 is diagnostic equivalence."
            ),

        "source_location":
            "Normalised exact comparison."
    },

    "reference_dataset_branch_independent":
        True,

    "comparison_rules_frozen_from_branch_A":
        True,

    "raw_extraction_modified":
        False,

    "manual_correction_applied":
        False,

    "reference_values_used_for_transformation":
        representation_integrity[
            "reference_values_used_for_transformation"
        ],

    "input_provenance":
        input_provenance
}

print(
    json.dumps(
        summary,
        indent=2,
        ensure_ascii=False
    )
)

{
  "document_id": "D5",
  "branch": "C",
  "branch_name": "Deterministic normalisation",
  "parent_branch": "B",
  "reference_records": 44,
  "extracted_records": 46,
  "aligned_records": 44,
  "fully_correct_records": 30,
  "discrepant_records": 14,
  "missing_records": 0,
  "unsupported_records": 2,
  "completeness": 1.0,
  "missing_rate": 0.0,
  "record_precision_exact": 0.6522,
  "record_recall_exact": 0.6818,
  "record_f1_exact": 0.6667,
  "unsupported_rate": 0.0435,
  "discrepancy_rate_among_aligned": 0.3182,
  "overall_primary_field_accuracy": 0.9513,
  "description_diagnostic_accuracy": 0.2955,
  "field_accuracy_among_aligned": {
    "Category": 1.0,
    "Indicator or Policy Area": 0.8636,
    "Value": 0.9091,
    "Unit": 0.9773,
    "Qualifier": 1.0,
    "Reference Period": 0.9091,
    "Description": 0.2955,
    "Source Location": 1.0
  },
  "schema_validity": true,
  "schema_diagnostics": {
    "valid_json": true,
    "top_level_object_valid": true,
    "document_id_present"

In [40]:
# ------------------------------------------------------------
# 16. Compact overall-results table
# ------------------------------------------------------------

overall_metrics_df = pd.DataFrame([
    {
        "metric":
            "Reference records",
        "value":
            N_REF
    },
    {
        "metric":
            "Extracted records",
        "value":
            N_EXT
    },
    {
        "metric":
            "Aligned records",
        "value":
            N_ALIGNED
    },
    {
        "metric":
            "Fully correct records",
        "value":
            N_CORRECT
    },
    {
        "metric":
            "Discrepant records",
        "value":
            N_DISCREPANT
    },
    {
        "metric":
            "Missing records",
        "value":
            N_MISSING
    },
    {
        "metric":
            "Unsupported records",
        "value":
            N_UNSUPPORTED
    },
    {
        "metric":
            "Completeness",
        "value":
            round(
                completeness,
                4
            )
    },
    {
        "metric":
            "Exact precision",
        "value":
            round(
                record_precision,
                4
            )
    },
    {
        "metric":
            "Exact recall",
        "value":
            round(
                record_recall,
                4
            )
    },
    {
        "metric":
            "Exact F1",
        "value":
            round(
                record_f1,
                4
            )
    },
    {
        "metric":
            "Overall primary-field accuracy",
        "value":
            round(
                overall_primary_field_accuracy,
                4
            )
    },
    {
        "metric":
            "Description diagnostic accuracy",
        "value":
            round(
                description_diagnostic_accuracy,
                4
            )
    },
    {
        "metric":
            "Unsupported rate",
        "value":
            round(
                unsupported_rate,
                4
            )
    },
    {
        "metric":
            "Schema validity",
        "value":
            schema_validity
    },
    {
        "metric":
            "Branch C normalisation integrity",
        "value":
            representation_integrity[
                "normalisation_integrity_passed"
            ]
    }
])

display(
    overall_metrics_df
)

,metric,value
0,Reference records,44
1,Extracted records,46
2,Aligned records,44
3,Fully correct records,30
4,Discrepant records,14
5,Missing records,0
6,Unsupported records,2
7,Completeness,1.0
8,Exact precision,0.6522
9,Exact recall,0.6818


In [41]:
# ------------------------------------------------------------
# 17. Validation integrity checks
# ------------------------------------------------------------

assert (
    N_ALIGNED
    + N_MISSING
    == N_REF
)

assert (
    N_ALIGNED
    + N_UNSUPPORTED
    == N_EXT
)

assert (
    N_CORRECT
    + N_DISCREPANT
    == N_ALIGNED
)

for metric_name, metric_value in {
    "completeness":
        completeness,

    "missing_rate":
        missing_rate,

    "record_precision":
        record_precision,

    "record_recall":
        record_recall,

    "record_f1":
        record_f1,

    "unsupported_rate":
        unsupported_rate,

    "discrepancy_rate":
        discrepancy_rate,

    "overall_primary_field_accuracy":
        overall_primary_field_accuracy,

    "description_diagnostic_accuracy":
        description_diagnostic_accuracy
}.items():

    assert (
        0.0
        <= metric_value
        <= 1.0
    ), (
        f"Invalid {metric_name}: "
        f"{metric_value}"
    )

print("Validation integrity checks passed.")

Validation integrity checks passed.


In [42]:
# ------------------------------------------------------------
# 18. Export validation artefacts
# ------------------------------------------------------------

comparison_df.to_csv(
    OUTPUT_DIR
    / "D5_branch_C_validation_detailed.csv",
    index=False,
    encoding="utf-8-sig"
)

field_validation_df.to_csv(
    OUTPUT_DIR
    / "D5_branch_C_field_validation.csv",
    index=False,
    encoding="utf-8-sig"
)

missing_records.to_csv(
    OUTPUT_DIR
    / "D5_branch_C_missing_records.csv",
    index=False,
    encoding="utf-8-sig"
)

unsupported_records.to_csv(
    OUTPUT_DIR
    / "D5_branch_C_unsupported_records.csv",
    index=False,
    encoding="utf-8-sig"
)

fully_correct_records.to_csv(
    OUTPUT_DIR
    / "D5_branch_C_fully_correct_records.csv",
    index=False,
    encoding="utf-8-sig"
)

discrepant_records.to_csv(
    OUTPUT_DIR
    / "D5_branch_C_discrepant_records.csv",
    index=False,
    encoding="utf-8-sig"
)

field_error_summary_df.to_csv(
    OUTPUT_DIR
    / "D5_branch_C_field_error_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

category_metrics_df.to_csv(
    OUTPUT_DIR
    / "D5_branch_C_category_metrics.csv",
    index=False,
    encoding="utf-8-sig"
)

overall_metrics_df.to_csv(
    OUTPUT_DIR
    / "D5_branch_C_overall_metrics.csv",
    index=False,
    encoding="utf-8-sig"
)

with open(
    OUTPUT_DIR
    / "D5_branch_C_validation_summary.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Validation artefacts saved.")

Validation artefacts saved.


In [43]:
# ------------------------------------------------------------
# 19. Final validation report
# ------------------------------------------------------------

final_report = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH_ID,

    "reference_records":
        N_REF,

    "extracted_records":
        N_EXT,

    "aligned_records":
        N_ALIGNED,

    "fully_correct_records":
        N_CORRECT,

    "discrepant_records":
        N_DISCREPANT,

    "missing_records":
        N_MISSING,

    "unsupported_records":
        N_UNSUPPORTED,

    "completeness":
        round(
            completeness,
            4
        ),

    "record_precision_exact":
        round(
            record_precision,
            4
        ),

    "record_recall_exact":
        round(
            record_recall,
            4
        ),

    "record_f1_exact":
        round(
            record_f1,
            4
        ),

    "overall_primary_field_accuracy":
        round(
            overall_primary_field_accuracy,
            4
        ),

    "schema_validity":
        schema_validity,

    "normalisation_integrity_passed":
        representation_integrity[
            "normalisation_integrity_passed"
        ],

    "comparison_rules_frozen_from_branch_A":
        True
}

print(
    json.dumps(
        final_report,
        indent=2,
        ensure_ascii=False
    )
)

{
  "document_id": "D5",
  "branch": "C",
  "reference_records": 44,
  "extracted_records": 46,
  "aligned_records": 44,
  "fully_correct_records": 30,
  "discrepant_records": 14,
  "missing_records": 0,
  "unsupported_records": 2,
  "completeness": 1.0,
  "record_precision_exact": 0.6522,
  "record_recall_exact": 0.6818,
  "record_f1_exact": 0.6667,
  "overall_primary_field_accuracy": 0.9513,
  "schema_validity": true,
  "normalisation_integrity_passed": true,
  "comparison_rules_frozen_from_branch_A": true
}


In [44]:
# ------------------------------------------------------------
# 20. Download generated validation artefacts
# ------------------------------------------------------------

for output_file in sorted(
    OUTPUT_DIR.iterdir()
):

    if output_file.is_file():

        print(
            "Downloading:",
            output_file.name
        )

        files.download(
            output_file
        )

Downloading: D5_branch_C_category_metrics.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloading: D5_branch_C_discrepant_records.csv


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloading: D5_branch_C_validation_summary.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>